# US Post-Market｜經濟數據與行事曆

行情、類股、個股與新聞改用 Bloomberg。這本筆記只做兩件事：最近公布的經濟數據（實際、預期、前期、意外），以及接下來的經濟行事曆（換成台北時間，標出晚上與凌晨要處理的事件）。全部執行約 5 秒。

## 1. Settings

In [1]:
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display

TW, NY = ZoneInfo("Asia/Taipei"), ZoneInfo("America/New_York")
TIMEOUT = 15
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}
AHEAD_DAYS = 10          # 行事曆往後看幾天
EVENING = (20, 24)       # 台北晚上的數據時段（美東 07:00–11:00 冬令）：當晚就能先寫
EARLY = (0, 5)           # 台北凌晨（FOMC 決議、會議紀錄、褐皮書在美東 14:00）：要提早起床

load_dotenv(".env")      # 金鑰放在同資料夾的 .env，不要寫進筆記本
FRED_API_KEY = os.getenv("FRED_API_KEY", "").strip()
print("FRED：", "已設定" if FRED_API_KEY else "未設定（實際值與下週以後的行事曆會缺）")

# 經濟數據：FRED 代號 → (中文, Forex Factory 標題, 算法)。算法：mom＝月增率（另附年增）、chg＝較前期增減（千人）、
# k＝以 K 表示（千）、m＝以 M 表示（百萬）、pct＝本身是百分比、lvl＝指數
ECON = {
    "CPIAUCSL": ("CPI", "CPI m/m", "mom"), "CPILFESL": ("核心 CPI", "Core CPI m/m", "mom"),
    "PPIFIS": ("PPI", "PPI m/m", "mom"), "PPIFES": ("核心 PPI", "Core PPI m/m", "mom"),
    "PCEPI": ("PCE 物價", "PCE Price Index m/m", "mom"), "PCEPILFE": ("核心 PCE", "Core PCE Price Index m/m", "mom"),
    "PAYEMS": ("非農就業", "Non-Farm Employment Change", "chg"), "UNRATE": ("失業率", "Unemployment Rate", "pct"),
    "CES0500000003": ("平均時薪", "Average Hourly Earnings m/m", "mom"),
    "ICSA": ("初領失業金", "Unemployment Claims", "k"), "CCSA": ("續領失業金", "", "k"),
    "JTSJOL": ("JOLTS 職缺", "JOLTS Job Openings", "m"),
    "RSAFS": ("零售銷售", "Retail Sales m/m", "mom"), "RSFSXMV": ("零售銷售（不含汽車）", "Core Retail Sales m/m", "mom"),
    "INDPRO": ("工業生產", "Industrial Production m/m", "mom"), "DGORDER": ("耐久財訂單", "Durable Goods Orders m/m", "mom"),
    "HOUST": ("新屋開工", "Housing Starts", "m"), "PERMIT": ("建築許可", "Building Permits", "m"),
    "HSN1F": ("新屋銷售", "New Home Sales", "k"), "UMCSENT": ("密大消費者信心", "UoM Consumer Sentiment", "lvl"),
    "A191RL1Q225SBEA": ("實質 GDP 季增年率", "GDP q/q", "pct"),
}
FRED_UNIT = {"ICSA": 1e3, "CCSA": 1e3, "HSN1F": 1, "JTSJOL": 1e3, "HOUST": 1e3, "PERMIT": 1e3}   # FRED 原始單位換成 K / M 的除數

# 下週以後的數據排程（FRED 發布日；FRED 沒有時間，用慣例的美東公布時間）。ISM、耐久財不在 FRED 排程裡
RELEASES = {10: ("CPI", "08:30"), 46: ("PPI", "08:30"), 50: ("非農就業報告", "08:30"), 54: ("個人所得與支出（PCE）", "08:30"),
            53: ("GDP", "08:30"), 9: ("零售銷售", "08:30"), 180: ("初領失業金", "08:30"), 27: ("新屋開工與建築許可", "08:30"),
            51: ("貿易收支", "08:30"), 188: ("進出口物價", "08:30"), 47: ("生產力與單位勞動成本", "08:30"),
            13: ("工業生產", "09:15"), 192: ("JOLTS 職缺", "10:00"), 97: ("新屋銷售", "10:00"), 91: ("密大消費者信心", "10:00"),
            229: ("營建支出", "10:00"), 194: ("ADP 就業", "08:15")}
# FOMC 利率決議日（美東，會議第二天，14:00 決議、14:30 記者會）。會議紀錄＝決議後 3 週的週三 14:00；褐皮書＝會前 2 週的週三 14:00。
# 每年到 Fed 網站更新：https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm（2027/10 的會議不表決利率）
FOMC = ["2026-01-28", "2026-03-18", "2026-04-29", "2026-06-17", "2026-07-29", "2026-09-16", "2026-10-28", "2026-12-09",
        "2027-01-27", "2027-03-17", "2027-04-28", "2027-06-09", "2027-07-28", "2027-09-15", "2027-10-27", "2027-11-17"]

FF_URL = "https://nfs.faireconomy.media/ff_calendar_thisweek.json"   # 本週全球行事曆（時間、預期值、前值；沒有實際值）
FF_CACHE, FF_TTL = "ff_calendar_cache.json", 1800                     # 對方限制頻率：30 分鐘內重跑讀快取
HTTP = requests.Session()
HTTP.headers.update(UA)


def fred(path, **params):
    if not FRED_API_KEY:
        raise ValueError("未設定 FRED_API_KEY")
    r = HTTP.get(f"https://api.stlouisfed.org/fred/{path}", params={**params, "api_key": FRED_API_KEY, "file_type": "json"},
                 timeout=TIMEOUT)
    if not r.ok:
        raise RuntimeError(f"FRED {r.status_code}")    # 金鑰在網址裡，不印出原始錯誤
    return r.json()


def ff_calendar():
    """Forex Factory 本週美國行事曆 → DataFrame(t 美東時間, title, impact, forecast, previous)"""
    if os.path.exists(FF_CACHE) and time.time() - os.path.getmtime(FF_CACHE) < FF_TTL:
        with open(FF_CACHE, encoding="utf-8") as f:
            data = json.load(f)
    else:
        r = HTTP.get(FF_URL, timeout=TIMEOUT)
        r.raise_for_status()
        data = r.json()
        with open(FF_CACHE, "w", encoding="utf-8") as f:
            json.dump(data, f)
    df = pd.DataFrame([e for e in data if e.get("country") == "USD"])
    df["t"] = [datetime.fromisoformat(d).astimezone(NY) for d in df["date"]]
    return df[["t", "title", "impact", "forecast", "previous"]].sort_values("t").reset_index(drop=True)


def prev_bday(d):
    d -= timedelta(days=1)
    while d.weekday() >= 5:
        d -= timedelta(days=1)
    return d


def tw(t):
    """美東時間 → 台北時間字串，例如「11/13（五）21:30」"""
    x = t.astimezone(TW)
    return f"{x:%m/%d}（{'一二三四五六日'[x.weekday()]}）{x:%H:%M}"


def slot(t):
    h = t.astimezone(TW).hour
    return "🌙 晚上" if EVENING[0] <= h < EVENING[1] else "⏰ 凌晨" if EARLY[0] <= h < EARLY[1] else ""


NOW = datetime.now(NY)
SINCE = prev_bday(NOW.date())     # 「最近公布」從前一個美國營業日算起
print(f"執行時間：台北 {NOW.astimezone(TW):%m/%d %H:%M}＝美東 {NOW:%m/%d %H:%M}｜最近公布從美東 {SINCE:%m/%d} 起算")

FRED： 已設定
執行時間：台北 11/13 17:36＝美東 11/13 04:36｜最近公布從美東 11/12 起算


## 2. 經濟數據（最近公布）

In [2]:
# 數據越強 → 升息預期越高：+1。初領失業金、失業率相反：-1
HAWK = {**{sid: 1 for sid in ECON}, "ICSA": -1, "CCSA": -1, "UNRATE": -1}
SPEAK = re.compile(r"(?:FOMC Member|Fed Chair|Fed Vice Chair|Fed Governor)\s+(\w+)\s+(?:Speaks|Testifies)")


def num(s):
    """'0.3%'、'194K'、'7.19M' → 0.3、194、7.19；空的 → None"""
    m = re.search(r"-?\d+(?:\.\d+)?", (s or "").replace(",", ""))
    return float(m.group()) if m else None


def base_title(t):
    return re.sub(r"^(Advance|Prelim|Preliminary|Final|Revised|Flash)\s+", "", t)


def released(sid):
    """FRED 在 SINCE 以後更新過 → 最新一期的實際值與前期；沒有 → None"""
    name, ff_title, kind = ECON[sid]
    meta = fred("series", series_id=sid)["seriess"][0]
    if pd.Timestamp(meta["last_updated"][:10]).date() < SINCE:
        return None
    obs = [(pd.Timestamp(o["date"]), float(o["value"])) for o in
           fred("series/observations", series_id=sid, sort_order="desc", limit=15)["observations"] if o["value"] != "."]
    (d0, v0), (d1, v1), (d2, v2) = obs[:3]
    freq = meta.get("frequency_short")
    period = f"{d0.year} Q{d0.quarter}" if freq == "Q" else f"{d0:%m/%d} 當週" if freq == "W" else f"{d0.year}/{d0.month}"
    note, div = "", FRED_UNIT.get(sid, 1)
    if kind == "mom":       # 中間缺月份（如政府停擺）時不算月增
        mom = lambda a, da, b, db: (a / b - 1) * 100 if da - pd.DateOffset(months=1) == db else None
        val, prior = mom(v0, d0, v1, d1), mom(v1, d1, v2, d2)
        yoy = next(((v0 / v - 1) * 100 for d, v in obs if d == d0 - pd.DateOffset(years=1)), None)
        note, fmt = (f"年增 {yoy:+.1f}%" if yoy is not None else ""), (lambda x: f"{x:+.1f}%")
    elif kind == "chg":
        val, prior, fmt = v0 - v1, v1 - v2, (lambda x: f"{x:+,.0f}K")
    elif kind == "k":
        val, prior, fmt = v0 / div, v1 / div, (lambda x: f"{x:,.0f}K")
    elif kind == "m":
        val, prior, fmt = v0 / div, v1 / div, (lambda x: f"{x:.2f}M")
    else:
        val, prior, fmt = v0, v1, (lambda x: f"{x:.1f}%" if kind == "pct" else f"{x:.1f}")
    show = lambda x: "—" if x is None else fmt(x)
    return dict(sid=sid, name=name, ff=ff_title, period=period, val=val, prior=show(prior), shown=show(val), note=note,
                digits=0 if kind in ("chg", "k") else 2 if kind == "m" else 1)


results, errs = {}, []
with ThreadPoolExecutor(6) as ex:
    futs = {sid: ex.submit(released, sid) for sid in ECON} if FRED_API_KEY else {}
for sid, f in futs.items():
    try:
        if (r := f.result()):
            results[sid] = r
    except Exception as e:
        errs.append(f"{ECON[sid][0]}（{type(e).__name__}）")
try:
    FF, ff_err = ff_calendar(), None
except Exception as e:
    FF, ff_err = pd.DataFrame(columns=["t", "title", "impact", "forecast", "previous"]), f"{type(e).__name__}"
recent = FF[(FF["t"].dt.date >= SINCE) & (FF["t"] <= NOW)]

lines = [f"### 經濟數據｜美東 {SINCE:%m/%d} 以來公布", ""]
if results:
    lines += ["| 數據 | 資料期 | 公布（台北） | 實際 | 預期 | 前期 | 意外 | 初步判讀 | 備註 |", "|:---|:---|:---|---:|---:|---:|---:|:---|:---|"]
    for sid, r in results.items():
        ev = recent[recent["title"].map(base_title) == r["ff"]] if r["ff"] else recent.iloc[0:0]
        when, fc = (tw(ev["t"].iloc[-1]), ev["forecast"].iloc[-1]) if len(ev) else ("—", "")
        f = num(fc)
        surprise, read = "—", ""
        if f is not None and r["val"] is not None:
            d = round(r["val"], r["digits"]) - f
            surprise = "符合" if abs(d) < 1e-9 else f"{d:+.{r['digits']}f}"
            if abs(d) >= 1e-9:
                hawk = (d > 0) == (HAWK[sid] > 0)
                read = ("強於預期" if d > 0 else "弱於預期") if sid not in ("ICSA", "CCSA", "UNRATE") else \
                       ("勞動市場轉弱" if d > 0 else "勞動市場偏強")
                read += " → 偏鷹（升息預期↑）" if hawk else " → 偏鴿（升息預期↓）"
        lines.append(f"| {r['name']} | {r['period']} | {when} | {r['shown']} | {fc or '—'} | {r['prior']} | {surprise} | {read} | {r['note']} |")
    lines += ["", "實際值、前期用 FRED 季調指數計算（和新聞稿可能差 0.1），前期是修正後的數字；預期值來自 Forex Factory。"
              "「初步判讀」只看數字和預期的差，實際要對照當天殖利率、升息定價怎麼反應（Bloomberg：WIRP、USGG2YR）。"]
else:
    lines.append("- 清單內的 FRED 數據這段期間都沒有公布" if FRED_API_KEY else "- 未設定 FRED_API_KEY")
if errs:
    lines.append(f"- ⚠️ FRED 無法取得：{'、'.join(errs)}")

covered = {r["ff"] for r in results.values()}
other = recent[(recent["forecast"].ne("") | recent["previous"].ne("")) & ~recent["title"].str.contains("Speaks|Testifies|Auction|Holiday")
               & ~recent["title"].map(base_title).isin(covered)]
if len(other):
    lines += ["", "**其他數據（FRED 沒有，實際值看 Bloomberg ECO）**", "", "| 公布（台北） | 項目 | 預期 | 前值 |", "|:---|:---|---:|---:|"]
    lines += [f"| {tw(e.t)} | {e.title} | {e.forecast or '—'} | {e.previous or '—'} |" for e in other.itertuples()]
speakers = [(m.group(1), e.t) for e in recent.itertuples() if (m := SPEAK.search(e.title))]
if speakers:
    lines += ["", "**Fed 官員談話（行程）**：" + "、".join(f"{who} {tw(t)}" for who, t in speakers)
              + "。盤勢段要把當天發言的官員列全；各自說了什麼看 Bloomberg 新聞"]
auctions = recent[recent["title"].str.contains("Auction")]
if len(auctions):
    lines += ["", "**公債標售**：" + "、".join(f"{e.title.replace(' Auction', '')} {tw(e.t)}（前次得標利率|投標倍數 {e.previous}）"
                                          for e in auctions.itertuples())]
if ff_err:
    lines.append(f"\n⚠️ Forex Factory 行事曆無法取得（{ff_err}）：沒有預期值與公布時間")
display(Markdown("\n".join(lines)))

### 經濟數據｜美東 11/12 以來公布

| 數據 | 資料期 | 公布（台北） | 實際 | 預期 | 前期 | 意外 | 初步判讀 | 備註 |
|:---|:---|:---|---:|---:|---:|---:|:---|:---|
| 初領失業金 | 11/07 當週 | 11/12（四）21:30 | 200K | 194K | 191K | +6 | 勞動市場轉弱 → 偏鴿（升息預期↓） |  |
| 續領失業金 | 10/31 當週 | — | 1,697K | — | 1,717K | — |  |  |

實際值、前期用 FRED 季調指數計算（和新聞稿可能差 0.1），前期是修正後的數字；預期值來自 Forex Factory。「初步判讀」只看數字和預期的差，實際要對照當天殖利率、升息定價怎麼反應（Bloomberg：WIRP、USGG2YR）。

**其他數據（FRED 沒有，實際值看 Bloomberg ECO）**

| 公布（台北） | 項目 | 預期 | 前值 |
|:---|:---|---:|---:|
| 11/12（四）23:00 | Existing Home Sales | 3.89M | 3.99M |
| 11/13（五）01:00 | Crude Oil Inventories | -0.6M | -0.9M |
| 11/13（五）03:00 | Federal Budget Balance | -298.6B | -26.9B |

**Fed 官員談話（行程）**：Hammack 11/12（四）21:30、Barr 11/12（四）23:00、Goolsbee 11/13（五）00:15、Daly 11/13（五）02:10、Schmid 11/13（五）02:30、Williams 11/13（五）07:30。盤勢段要把當天發言的官員列全；各自說了什麼看 Bloomberg 新聞

**公債標售**：30-y Bond 11/13（五）02:01（前次得標利率|投標倍數 5.62|2.5）

## 3. 經濟行事曆

In [3]:
END = NOW + timedelta(days=AHEAD_DAYS)
KEY = re.compile(r"CPI|PPI|Non-Farm|PCE|GDP|Retail Sales|ISM|Federal Funds Rate|FOMC (Statement|Press|Meeting Minutes)|Beige Book|"
                 r"Fed Chair|JOLTS")
ZH = {"Federal Funds Rate": "FOMC 利率決議", "FOMC Statement": "FOMC 聲明", "FOMC Press Conference": "FOMC 記者會",
      "FOMC Meeting Minutes": "FOMC 會議紀錄", "Beige Book": "褐皮書", "Bank Holiday": "銀行假日（債市休市；股市照常與否另查）"}
ZH.update({ff: name for name, ff, _ in ECON.values() if ff})
at = lambda d, hm: datetime.combine(d, datetime.strptime(hm, "%H:%M").time(), NY)


def label(title):
    if (m := SPEAK.search(title)):
        return f"Fed 官員談話：{m.group(1)}"
    if (m := re.match(r"(\d+)-y (Bond|Note) Auction", title)):
        return f"{m.group(1)} 年期公債標售"
    head = re.match(r"^(Advance|Prelim|Preliminary|Final|Revised|Flash)\s+", title)
    zh = ZH.get(base_title(title))
    return f"{zh}（{head.group(1)}）" if zh and head else zh or title


rows = [dict(t=e.t, item=label(e.title), fc=e.forecast, prev=e.previous, star=e.impact == "High" or bool(KEY.search(e.title)),
             mid=e.impact == "Medium")
        for e in FF[(FF["t"] > NOW) & (FF["t"] <= END)].itertuples()]
# Forex Factory 只有本週（週日到週六）；之後的用 FRED 發布日、TreasuryDirect 標售排程、FOMC 日程補上
ff_end = (FF["t"].min() + timedelta(days=(5 - FF["t"].min().weekday()) % 7)).date() if len(FF) else NOW.date() - timedelta(days=1)
later, notes = lambda d: ff_end < d <= END.date(), []
if FRED_API_KEY:
    try:
        rd = fred("releases/dates", realtime_start=f"{NOW:%Y-%m-%d}", realtime_end=f"{END:%Y-%m-%d}", limit=1000,
                  include_release_dates_with_no_data="true", order_by="release_date", sort_order="asc")["release_dates"]
        rows += [dict(t=at(d, RELEASES[x["release_id"]][1]), item=RELEASES[x["release_id"]][0], fc="", prev="",
                      star=x["release_id"] in (10, 46, 50, 54, 53, 9))
                 for x in rd if x["release_id"] in RELEASES and later(d := date.fromisoformat(x["date"]))]
    except Exception as e:
        notes.append(f"FRED 發布日無法取得（{type(e).__name__}）")
try:
    r = HTTP.get("https://www.treasurydirect.gov/TA_WS/securities/upcoming", params={"format": "json"}, timeout=TIMEOUT)
    for a in r.json():
        d = (a.get("auctionDate") or a.get("competitiveAuctionCloseDate") or "")[:10]
        if a.get("securityType") in ("Note", "Bond", "TIPS", "FRN") and d and later(date.fromisoformat(d)):
            amt = f"（{float(a['offeringAmount']) / 1e8:,.0f} 億美元）" if a.get("offeringAmount") else ""
            rows.append(dict(t=at(date.fromisoformat(d), "13:00"), item=f"{a['securityTerm']} {a['securityType']} 標售{amt}",
                             fc="", prev="", star=False))
except Exception as e:
    notes.append(f"TreasuryDirect 標售排程無法取得（{type(e).__name__}）")
for d in map(date.fromisoformat, FOMC):
    for when, item in ((d, "FOMC 利率決議（14:30 記者會）"), (d + timedelta(days=21), "FOMC 會議紀錄（推算）"),
                       (d - timedelta(days=14), "褐皮書（推算）")):
        if later(when):
            rows.append(dict(t=at(when, "14:00"), item=item, fc="", prev="", star=True))

rows.sort(key=lambda x: x["t"])
today_tw = NOW.astimezone(TW).date()
tonight = [x for x in rows if x["t"].astimezone(TW).date() == today_tw and slot(x["t"]) == "🌙 晚上"
           and "談話" not in x["item"] and (x["star"] or x.get("mid"))]
early = [x for x in rows if slot(x["t"]) == "⏰ 凌晨" and re.search(r"FOMC|褐皮書|Powell|Warsh", x["item"])]
lines = [f"### 經濟行事曆｜台北 {NOW.astimezone(TW):%m/%d %H:%M} 起 {AHEAD_DAYS} 天", "",
         "- 今晚（台北 20:00–24:00）的數據：" + ("、".join(f"{x['item']} {x['t'].astimezone(TW):%H:%M}" for x in tonight) or "沒有"),
         "- 要凌晨起床處理的 Fed 事件：" + ("、".join(f"{x['item']} {tw(x['t'])}" for x in early) or "沒有"), "",
         "| 台北時間 | 時段 | 事件 | 預期 | 前值 |", "|:---|:---|:---|---:|---:|"]
lines += [f"| {tw(x['t'])} | {slot(x['t'])} | {'⭐ ' if x['star'] else ''}{x['item']} | {x['fc'] or ''} | {x['prev'] or ''} |" for x in rows]
lines += ["", f"本週（到美東 {ff_end:%m/%d}）來自 Forex Factory，含時間、預期與前值；之後的數據用 FRED 發布日加慣例時間，"
          "標售用 TreasuryDirect 已公告的排程，FOMC 會議紀錄與褐皮書是推算的日期。ISM、耐久財、消費者信心等下週以後的項目以 Bloomberg ECO 為準。"
          "⭐＝高影響。🌙＝台北晚上公布，當晚可以先寫；⏰＝台北凌晨，要提早起床。"]
if notes:
    lines.append("⚠️ " + "；".join(notes))
display(Markdown("\n".join(lines)))

### 經濟行事曆｜台北 11/13 17:36 起 10 天

- 今晚（台北 20:00–24:00）的數據：核心 PPI 21:30、PPI 21:30
- 要凌晨起床處理的 Fed 事件：FOMC 會議紀錄（推算） 11/19（四）03:00

| 台北時間 | 時段 | 事件 | 預期 | 前值 |
|:---|:---|:---|---:|---:|
| 11/13（五）21:30 | 🌙 晚上 | ⭐ 核心 PPI | 0.3% | 0.7% |
| 11/13（五）21:30 | 🌙 晚上 | ⭐ PPI | 0.4% | 1.3% |
| 11/13（五）22:10 | 🌙 晚上 | Fed 官員談話：Collins |  |  |
| 11/13（五）22:40 | 🌙 晚上 | Fed 官員談話：Logan |  |  |
| 11/13（五）23:15 | 🌙 晚上 | Mortgage Delinquencies |  | 4.37% |
| 11/13（五）23:30 | 🌙 晚上 | Natural Gas Storage | 65B | 77B |
| 11/13（五）23:50 | 🌙 晚上 | Fed 官員談話：Bowman |  |  |
| 11/14（六）01:00 | ⏰ 凌晨 | Fed 官員談話：Daly |  |  |
| 11/14（六）08:00 |  | Fed 官員談話：Jefferson |  |  |
| 11/17（二）21:30 | 🌙 晚上 | ⭐ 零售銷售 |  |  |
| 11/17（二）21:30 | 🌙 晚上 | 進出口物價 |  |  |
| 11/17（二）22:15 | 🌙 晚上 | 工業生產 |  |  |
| 11/18（三）21:30 | 🌙 晚上 | 新屋開工與建築許可 |  |  |
| 11/19（四）02:00 | ⏰ 凌晨 | 20-Year Bond 標售（160 億美元） |  |  |
| 11/19（四）03:00 | ⏰ 凌晨 | ⭐ FOMC 會議紀錄（推算） |  |  |
| 11/19（四）21:30 | 🌙 晚上 | 初領失業金 |  |  |
| 11/20（五）02:00 | ⏰ 凌晨 | 9-Year 8-Month Note 標售（190 億美元） |  |  |
| 11/20（五）23:00 | 🌙 晚上 | 密大消費者信心 |  |  |

本週（到美東 11/14）來自 Forex Factory，含時間、預期與前值；之後的數據用 FRED 發布日加慣例時間，標售用 TreasuryDirect 已公告的排程，FOMC 會議紀錄與褐皮書是推算的日期。ISM、耐久財、消費者信心等下週以後的項目以 Bloomberg ECO 為準。⭐＝高影響。🌙＝台北晚上公布，當晚可以先寫；⏰＝台北凌晨，要提早起床。